In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA health")

df_v2 = spark.table("gold_mart_patient_day_v2")
df_v2.printSchema()

df_v2.select(
    "patient_id", "date",
    "avg_sys", "avg_dia", "cholesterol", "glucose", "age_est",
    "risk_label", "risk_label_base", "risk_label_v2",
    "risk_score_base", "risk_score_v2"
).show(20, truncate=False)


In [0]:
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# 1) Đọc bảng Gold v2
df_v2 = spark.table("gold_mart_patient_day_v2")

# 2) Chọn các cột dùng cho ML
num_features = [
    "avg_hr",
    "avg_sys",
    "avg_dia",
    "avg_spo2",
    "cholesterol",
    "glucose",
    "hemoglobin",
    "age_est"
]
cat_features = ["gender", "region"]

df_ml = df_v2.select(
    ["patient_id", "date", "risk_label_v2"] + num_features + cat_features
)

# 3) Drop các dòng thiếu dữ liệu
df_ml = df_ml.dropna()

print("Rows:", df_ml.count())
df_ml.show(5, truncate=False)

# 4) Train-test split
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)
print("Train:", train_df.count(), " Test:", test_df.count())


In [0]:
def eval_multiclass(pred_df, labelCol="label", predictionCol="prediction"):
    evaluator = MulticlassClassificationEvaluator(
        labelCol=labelCol,
        predictionCol=predictionCol
    )
    return {
        "accuracy":  evaluator.setMetricName("accuracy").evaluate(pred_df),
        "f1":         evaluator.setMetricName("f1").evaluate(pred_df),
        "precision":  evaluator.setMetricName("weightedPrecision").evaluate(pred_df),
        "recall":     evaluator.setMetricName("weightedRecall").evaluate(pred_df),
    }


In [0]:
# 1) Index label + categorical
label_indexer = StringIndexer(
    inputCol="risk_label_v2",
    outputCol="label",
    handleInvalid="keep"
)

gender_indexer = StringIndexer(
    inputCol="gender",
    outputCol="gender_idx",
    handleInvalid="keep"
)

region_indexer = StringIndexer(
    inputCol="region",
    outputCol="region_idx",
    handleInvalid="keep"
)

onehot = OneHotEncoder(
    inputCols=["gender_idx", "region_idx"],
    outputCols=["gender_oh", "region_oh"]
)

# 2) Assemble + scale
assembler_lr = VectorAssembler(
    inputCols=num_features + ["gender_oh", "region_oh"],
    outputCol="features"
)

scaler = StandardScaler(
    inputCol="features",
    outputCol="features_scaled",
    withStd=True,
    withMean=False
)

# 3) Model Logistic Regression
lr = LogisticRegression(
    featuresCol="features_scaled",
    labelCol="label",
    maxIter=50,
    regParam=0.0,
    elasticNetParam=0.0
)

pipeline_lr = Pipeline(stages=[
    label_indexer,
    gender_indexer,
    region_indexer,
    onehot,
    assembler_lr,
    scaler,
    lr
])

# 4) Train + predict
lr_model = pipeline_lr.fit(train_df)
pred_lr = lr_model.transform(test_df)

metrics_lr = eval_multiclass(pred_lr)
print("=== CLEAN v2 – Logistic Regression ===")
for k, v in metrics_lr.items():
    print(f"{k}: {v:.4f}")


In [0]:
# CELL LR-VIZ-0 — Prepare pred_lr_v (labels + prob columns)

from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array
from pyspark.ml.feature import StringIndexerModel

LABEL_INPUT_COL = "risk_label_v2"   # label gốc (string)
LABEL_IDX_COL   = "label"          # label sau StringIndexer
PROB_COL        = "probability"    # Spark ML output
PRED_COL        = "prediction"

# 1) Lấy label order đúng từ StringIndexerModel (stage label_indexer trong pipeline)
label_model = None
for st in lr_model.stages:
    if isinstance(st, StringIndexerModel) and st.getInputCol() == LABEL_INPUT_COL:
        label_model = st
        break

if label_model is None:
    raise ValueError("Không tìm thấy StringIndexerModel cho risk_label_v2 trong lr_model.stages")

labels = label_model.labels                  # ví dụ: ['Elevated','High','Normal'] (thứ tự này quyết định prob_0/1/2)
labels_arr = F.array(*[F.lit(x) for x in labels])

# 2) Convert prob vector -> array; map label_true/pred theo đúng label order
pred_lr_v = (
    pred_lr
    .withColumn("prob_arr", vector_to_array(F.col(PROB_COL)))
    .withColumn("label_int", F.col(LABEL_IDX_COL).cast("int"))
    .withColumn("pred_int",  F.col(PRED_COL).cast("int"))
    .withColumn("label_true", F.element_at(labels_arr, F.col("label_int") + F.lit(1)))
    .withColumn("label_pred", F.element_at(labels_arr, F.col("pred_int")  + F.lit(1)))
    .withColumn("prob_0", F.col("prob_arr")[0])
    .withColumn("prob_1", F.col("prob_arr")[1])
    .withColumn("prob_2", F.col("prob_arr")[2])
)

print("Label order (StringIndexer):", labels)
pred_lr_v.select(LABEL_INPUT_COL, "label_true", "label_pred", "prob_0", "prob_1", "prob_2").show(5, truncate=False)


In [0]:
# CELL LR-VIZ-1 — Confusion Matrix (LR Test)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cm_long = (pred_lr_v
    .groupBy("label_true", "label_pred")
    .count()
)

cm_pdf = (cm_long
    .toPandas()
    .pivot(index="label_true", columns="label_pred", values="count")
    .reindex(index=labels, columns=labels)
    .fillna(0)
)

plt.figure()
plt.imshow(cm_pdf.values)
plt.title("Logistic Regression — Confusion Matrix (TEST)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(range(len(labels)), labels)
plt.yticks(range(len(labels)), labels)

for i in range(len(labels)):
    for j in range(len(labels)):
        plt.text(j, i, int(cm_pdf.values[i, j]), ha="center", va="center")

plt.show()

total = int(cm_pdf.values.sum())
correct = int(np.trace(cm_pdf.values))
acc = correct / total if total > 0 else 0.0
print("LR TEST accuracy:", round(acc, 4), f"({correct}/{total})")


In [0]:
# CELL LR-VIZ-2 — Per-class Metrics Bar (LR Test)

def per_class_metrics_from_cm(cm: pd.DataFrame):
    arr = cm.values.astype(float)
    rows = []
    for k, cls in enumerate(cm.index.tolist()):
        tp = arr[k, k]
        fp = arr[:, k].sum() - tp
        fn = arr[k, :].sum() - tp

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

        rows.append([cls, precision, recall, f1, int(arr[k, :].sum())])

    return pd.DataFrame(rows, columns=["class","precision","recall","f1","support"])

per_cls = per_class_metrics_from_cm(cm_pdf)
display(per_cls)

import numpy as np
import matplotlib.pyplot as plt

plot_df = per_cls.set_index("class").reindex(labels).reset_index()

x = np.arange(len(labels))
w = 0.25

plt.figure()
plt.bar(x - w, plot_df["precision"].values, width=w, label="Precision")
plt.bar(x,     plot_df["recall"].values,    width=w, label="Recall")
plt.bar(x + w, plot_df["f1"].values,        width=w, label="F1")

plt.xticks(x, labels)
plt.ylim(0, 1.0)
plt.title("Per-class Metrics — Logistic Regression (TEST)")
plt.ylabel("Score")
plt.legend()
plt.show()

print("Macro F1:", round(per_cls["f1"].mean(), 4))
print("Weighted F1:", round((per_cls["f1"] * per_cls["support"]).sum() / per_cls["support"].sum(), 4))


In [0]:
# CELL LR-VIZ-3 — ROC Curves (OvR) — Logistic Regression (TEST)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

roc_pdf = (pred_lr_v
    .select("label_true", "prob_0", "prob_1", "prob_2")
    .sample(False, 0.30, seed=42)   # chỉnh 0.1..1.0 tuỳ dữ liệu
    .toPandas()
)

y_true_txt = roc_pdf["label_true"].astype(str).values
y_true_bin = label_binarize(y_true_txt, classes=labels)  # (n,3)

proba = roc_pdf[["prob_0","prob_1","prob_2"]].astype(float).values

plt.figure()
for i, cls in enumerate(labels):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], proba[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{cls} (AUC={roc_auc:.3f})")

plt.plot([0,1], [0,1], linestyle="--", label="Random")
plt.title("ROC Curves (OvR) — Logistic Regression (TEST)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()


In [0]:
# CELL LR-VIZ-4 — Class Distribution Train vs Test (LR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

train_cnt = (train_df.groupBy("risk_label_v2").count().toPandas()
             .set_index("risk_label_v2")["count"])
test_cnt  = (test_df.groupBy("risk_label_v2").count().toPandas()
             .set_index("risk_label_v2")["count"])

train_cnt = train_cnt.reindex(labels).fillna(0).astype(int)
test_cnt  = test_cnt.reindex(labels).fillna(0).astype(int)

x = np.arange(len(labels))
w = 0.35

plt.figure()
plt.bar(x - w/2, train_cnt.values, width=w, label="Train")
plt.bar(x + w/2, test_cnt.values,  width=w, label="Test")
plt.xticks(x, labels)
plt.title("Class Distribution — Train vs Test (Logistic Regression)")
plt.ylabel("Count")
plt.legend()
plt.show()


In [0]:
# CELL LR-VIZ-5 — Probability distribution for true class (LR)

import matplotlib.pyplot as plt

# sample cho nhẹ
sample_pdf = (pred_lr_v
    .select("label_true", "prob_0", "prob_1", "prob_2")
    .sample(False, 0.25, seed=42)
    .toPandas()
)

# với class = labels[i] thì xác suất P(class) nằm ở prob_i
for i, cls in enumerate(labels):
    col = f"prob_{i}"
    plt.figure()
    plt.hist(sample_pdf.loc[sample_pdf["label_true"] == cls, col].astype(float), bins=25)
    plt.title(f"LR — Probability distribution (true class = {cls})")
    plt.xlabel(f"P({cls})")
    plt.ylabel("count")
    plt.show()


In [0]:
# CELL LR-VIZ-6 — Logistic Regression Coefficients (Top 25)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.ml.classification import LogisticRegressionModel
from pyspark.ml.feature import OneHotEncoderModel, StringIndexerModel
from pyspark.ml.feature import VectorAssembler

# 1) tìm LR model stage trong pipeline
lr_stage = None
for st in lr_model.stages:
    if isinstance(st, LogisticRegressionModel):
        lr_stage = st
        break
if lr_stage is None:
    raise ValueError("Không tìm thấy LogisticRegressionModel trong lr_model.stages")

# 2) xác định VectorAssembler tạo features (trong pipeline_lr bạn đặt outputCol='features', sau đó scaler -> 'features_scaled')
# hệ số nằm trên features_scaled, nhưng thứ tự feature vẫn theo vector trước scale.
assembler = None
for st in lr_model.stages:
    if isinstance(st, VectorAssembler) and st.getOutputCol() == "features":
        assembler = st
        break

# 3) Build feature names: numeric + expanded onehot (giống RF)
if assembler is None:
    feature_names = [f"f{i}" for i in range(lr_stage.numFeatures)]
else:
    input_cols = assembler.getInputCols()

    # find OHE model (nếu có)
    ohe = None
    for st in lr_model.stages:
        if isinstance(st, OneHotEncoderModel):
            ohe = st
            break

    # map idxCol -> labels
    idx_labels = {}
    for st in lr_model.stages:
        if isinstance(st, StringIndexerModel):
            idx_labels[st.getOutputCol()] = st.labels

    def expand_ohe_names(ohe_model: OneHotEncoderModel):
        in_cols  = ohe_model.getInputCols()
        out_cols = ohe_model.getOutputCols()
        drop_last = ohe_model.getDropLast()

        expanded = {}
        for in_c, out_c, size in zip(in_cols, out_cols, ohe_model.categorySizes):
            labs = idx_labels.get(in_c, [f"{in_c}_{i}" for i in range(size)])
            labs_use = labs[:-1] if (drop_last and len(labs) > 0) else labs
            expanded[out_c] = [f"{in_c.replace('_idx','')}={lab}" for lab in labs_use]
        return expanded

    expanded_ohe = expand_ohe_names(ohe) if ohe is not None else {}

    feature_names = []
    for c in input_cols:
        if c in expanded_ohe:
            feature_names.extend(expanded_ohe[c])
        else:
            feature_names.append(c)

    if len(feature_names) != lr_stage.numFeatures:
        print(f"[WARN] feature_names({len(feature_names)}) != numFeatures({lr_stage.numFeatures}). Fallback f0..fN")
        feature_names = [f"f{i}" for i in range(lr_stage.numFeatures)]

# 4) lấy coefficient (multiclass => coefficientMatrix)
coef_mat = lr_stage.coefficientMatrix.toArray()  # shape (numClasses, numFeatures)
# importance tổng hợp: norm theo class hoặc max abs
importance = np.linalg.norm(coef_mat, axis=0)     # L2 norm across classes

fi = (pd.DataFrame({"feature": feature_names, "importance": importance})
      .sort_values("importance", ascending=False)
      .head(25))

display(fi)

plt.figure()
plt.barh(fi["feature"][::-1], fi["importance"][::-1])
plt.title("LR — Coefficient Importance (Top 25) [L2 norm across classes]")
plt.xlabel("importance")
plt.ylabel("feature")
plt.show()


In [0]:
cm_lr = (pred_lr
    .groupBy("risk_label_v2", "prediction")
    .count()
    .groupBy("risk_label_v2")
    .pivot("prediction")
    .sum("count")
    .orderBy("risk_label_v2")
)

cm_lr.show(truncate=False)


In [0]:
# Dùng lại label_indexer, gender_indexer, region_indexer, onehot

assembler_rf = VectorAssembler(
    inputCols=num_features + ["gender_oh", "region_oh"],
    outputCol="features_rf"
)

rf = RandomForestClassifier(
    featuresCol="features_rf",
    labelCol="label",
    numTrees=100,
    maxDepth=8,
    seed=42
)

pipeline_rf = Pipeline(stages=[
    label_indexer,
    gender_indexer,
    region_indexer,
    onehot,
    assembler_rf,
    rf
])

rf_model = pipeline_rf.fit(train_df)
pred_rf = rf_model.transform(test_df)

metrics_rf = eval_multiclass(pred_rf)
print("\n=== CLEAN v2 – Random Forest ===")
for k, v in metrics_rf.items():
    print(f"{k}: {v:.4f}")


In [0]:
cm_rf = (pred_rf
    .groupBy("risk_label_v2", "prediction")
    .count()
    .groupBy("risk_label_v2")
    .pivot("prediction")
    .sum("count")
    .orderBy("risk_label_v2")
)

cm_rf.show(truncate=False)


In [0]:
from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array

FEATURES_COL = "features_rf"
LABEL_IDX_COL = "label"

# 1) Lấy label mapping đúng từ stage StringIndexer của risk_label_v2
label_model = None
for st in rf_model.stages:
    if hasattr(st, "getInputCol") and st.getInputCol() == "risk_label_v2":
        label_model = st
        break

if label_model is None:
    raise ValueError("Không tìm thấy StringIndexerModel cho inputCol='risk_label_v2' trong rf_model.stages")

labels = label_model.labels  # ví dụ: ['Elevated','High','Normal'] hoặc khác
labels_arr = F.array(*[F.lit(x) for x in labels])

# 2) Convert probability vector -> array để lấy từng class prob
pred_rf_v = (pred_rf
    .withColumn("prob_arr", vector_to_array(F.col("probability")))
    .withColumn("label_int", F.col(LABEL_IDX_COL).cast("int"))
    .withColumn("pred_int", F.col("prediction").cast("int"))
    .withColumn("label_true", F.element_at(labels_arr, F.col("label_int") + F.lit(1)))
    .withColumn("label_pred", F.element_at(labels_arr, F.col("pred_int") + F.lit(1)))
    .withColumn("prob_0", F.col("prob_arr")[0])
    .withColumn("prob_1", F.col("prob_arr")[1])
    .withColumn("prob_2", F.col("prob_arr")[2])
)

pred_rf_v.select("risk_label_v2","label_true","label_pred","prob_0","prob_1","prob_2").show(5, truncate=False)
print("Label order (theo StringIndexer):", labels)


In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Confusion matrix counts trong Spark
cm_long = (pred_rf_v
    .groupBy("label_true", "label_pred")
    .count()
)

cm_pdf = (cm_long
    .toPandas()
    .pivot(index="label_true", columns="label_pred", values="count")
    .reindex(index=labels, columns=labels)
    .fillna(0)
)

# Plot confusion matrix
plt.figure()
plt.imshow(cm_pdf.values)
plt.title("RF - Confusion Matrix (TEST)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(range(len(labels)), labels, rotation=0)
plt.yticks(range(len(labels)), labels)

for i in range(len(labels)):
    for j in range(len(labels)):
        plt.text(j, i, int(cm_pdf.values[i, j]), ha="center", va="center")

plt.show()

# Accuracy
total = int(cm_pdf.values.sum())
correct = int(np.trace(cm_pdf.values))
acc = correct / total if total > 0 else 0.0
print("RF TEST accuracy:", round(acc, 4), f"({correct}/{total})")


In [0]:
# CELL RF-VIZ-ROC — ROC Curves (OvR) — Random Forest (Test)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

# 1) Lấy dữ liệu test (có thể sample để nhẹ máy)
roc_pdf = (pred_rf_v
    .select("label_true", "prob_0", "prob_1", "prob_2")
    .sample(False, 0.30, seed=42)   # chỉnh 0.1-1.0 tuỳ dữ liệu
    .toPandas()
)

# 2) y_true binarize theo đúng thứ tự labels (labels = order của prob_0/1/2)
y_true_txt = roc_pdf["label_true"].astype(str).values
y_true_bin = label_binarize(y_true_txt, classes=labels)  # shape (n,3)

proba = roc_pdf[["prob_0","prob_1","prob_2"]].astype(float).values

# 3) Plot ROC OvR
plt.figure()

for i, cls in enumerate(labels):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], proba[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{cls} (AUC={roc_auc:.3f})")

plt.plot([0,1], [0,1], linestyle="--", label="Random")
plt.title("ROC Curves (OvR) — Random Forest (Test)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()


In [0]:
# CELL RF-VIZ-PERCLASS-BAR — Per-class Metrics (RF Test)

import numpy as np
import matplotlib.pyplot as plt

# đảm bảo thứ tự theo labels
plot_df = per_cls.set_index("class").reindex(labels).reset_index()

x = np.arange(len(labels))
w = 0.25

plt.figure()
plt.bar(x - w, plot_df["precision"].values, width=w, label="Precision")
plt.bar(x,     plot_df["recall"].values,    width=w, label="Recall")
plt.bar(x + w, plot_df["f1"].values,        width=w, label="F1")

plt.xticks(x, labels)
plt.ylim(0, 1.0)
plt.title("Per-class Metrics — Random Forest (Test)")
plt.ylabel("Score")
plt.legend()
plt.show()


In [0]:
# CELL RF-VIZ-CLASSDIST — Class Distribution Train vs Test (RF)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Count theo risk_label_v2 trong train/test
train_cnt = (train_df.groupBy("risk_label_v2").count().toPandas()
             .set_index("risk_label_v2")["count"])
test_cnt  = (test_df.groupBy("risk_label_v2").count().toPandas()
             .set_index("risk_label_v2")["count"])

# reindex theo labels (thứ tự đang dùng trong RF)
train_cnt = train_cnt.reindex(labels).fillna(0).astype(int)
test_cnt  = test_cnt.reindex(labels).fillna(0).astype(int)

x = np.arange(len(labels))
w = 0.35

plt.figure()
plt.bar(x - w/2, train_cnt.values, width=w, label="Train")
plt.bar(x + w/2, test_cnt.values,  width=w, label="Test")
plt.xticks(x, labels)
plt.title("Class Distribution — Train vs Test (RF)")
plt.ylabel("Count")
plt.legend()
plt.show()


In [0]:
def per_class_metrics_from_cm(cm: pd.DataFrame):
    arr = cm.values.astype(float)
    metrics = []
    for k, cls in enumerate(cm.index.tolist()):
        tp = arr[k, k]
        fp = arr[:, k].sum() - tp
        fn = arr[k, :].sum() - tp

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

        metrics.append([cls, precision, recall, f1, int(arr[k, :].sum())])

    return pd.DataFrame(metrics, columns=["class","precision","recall","f1","support"])

per_cls = per_class_metrics_from_cm(cm_pdf)
display(per_cls)

print("Macro F1:", round(per_cls["f1"].mean(), 4))
print("Weighted F1:", round((per_cls["f1"] * per_cls["support"]).sum() / per_cls["support"].sum(), 4))


In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.ml.feature import StringIndexerModel, OneHotEncoderModel, VectorAssembler

FEATURES_COL = "features_rf"   # giữ đúng như pipeline bạn đang dùng

# 1) lấy RF stage (thường stage cuối)
rf_stage = rf_model.stages[-1]
importances = np.array(rf_stage.featureImportances.toArray())

# 2) tìm VectorAssembler tạo ra FEATURES_COL
assembler = None
for st in rf_model.stages:
    if isinstance(st, VectorAssembler) and st.getOutputCol() == FEATURES_COL:
        assembler = st
        break

if assembler is None:
    # fallback: không tìm được assembler -> dùng f0..fN
    feature_names = [f"f{i}" for i in range(len(importances))]
else:
    input_cols = assembler.getInputCols()

    # 3) tìm OneHotEncoderModel (nếu có) và mapping labels từ StringIndexerModel
    ohe = None
    for st in rf_model.stages:
        if isinstance(st, OneHotEncoderModel):
            ohe = st
            break

    # map: inputCol -> labels (từ StringIndexerModel)
    idx_labels = {}
    for st in rf_model.stages:
        if isinstance(st, StringIndexerModel):
            idx_labels[st.getOutputCol()] = st.labels  # outputCol (vd gender_idx) -> labels list

    # helper: expand onehot vector feature names
    def expand_ohe_names(ohe_model: OneHotEncoderModel):
        # ohe inputs/outputs
        in_cols  = ohe_model.getInputCols()
        out_cols = ohe_model.getOutputCols()
        drop_last = ohe_model.getDropLast()

        expanded = {}
        for in_c, out_c, size in zip(in_cols, out_cols, ohe_model.categorySizes):
            # in_c thường là *_idx (vd gender_idx)
            labels = idx_labels.get(in_c, [f"{in_c}_{i}" for i in range(size)])
            # nếu dropLast=True thì bỏ category cuối
            if drop_last and len(labels) > 0:
                labels_use = labels[:-1]
            else:
                labels_use = labels

            expanded[out_c] = [f"{in_c.replace('_idx','')}={lab}" for lab in labels_use]
        return expanded

    expanded_ohe = expand_ohe_names(ohe) if ohe is not None else {}

    # 4) build feature_names theo đúng thứ tự assembler input
    feature_names = []
    for c in input_cols:
        if c in expanded_ohe:
            feature_names.extend(expanded_ohe[c])   # vector expand
        else:
            feature_names.append(c)                 # numeric (hoặc col thường)

    # 5) nếu vẫn lệch độ dài, fallback an toàn
    if len(feature_names) != len(importances):
        print(f"[WARN] feature_names({len(feature_names)}) != importances({len(importances)}). Fallback to f0..fN.")
        feature_names = [f"f{i}" for i in range(len(importances))]

# 6) Top 25 importance
fi = (pd.DataFrame({"feature": feature_names, "importance": importances})
      .sort_values("importance", ascending=False)
      .head(25))

display(fi)

plt.figure()
plt.barh(fi["feature"][::-1], fi["importance"][::-1])
plt.title("RF - Top 25 Feature Importances")
plt.xlabel("importance")
plt.ylabel("feature")
plt.show()


In [0]:
# Lấy sample nhỏ để vẽ histogram cho nhẹ (có thể tăng lên)
sample_pdf = (pred_rf_v
    .select("label_true","prob_0","prob_1","prob_2")
    .sample(False, 0.2, seed=42)
    .toPandas()
)

# Map prob theo đúng class name (labels[0], labels[1], labels[2])
for i, cls in enumerate(labels):
    col = f"prob_{i}"
    plt.figure()
    plt.hist(sample_pdf.loc[sample_pdf["label_true"] == cls, col].astype(float), bins=20)
    plt.title(f"RF - Probability distribution for true class = {cls}")
    plt.xlabel(f"P({cls})")
    plt.ylabel("count")
    plt.show()


In [0]:
print([type(s).__name__ for s in rf_model.stages])
print("Assembler inputCols:", [s.getInputCols() for s in rf_model.stages if hasattr(s, "getInputCols") and hasattr(s, "getOutputCol") and s.getOutputCol()=="features_rf"])
